In [1]:
# Step 1: Load only Chapter 5 from the textbook
# Chapter 5 = pages 96–119 in the printed book
# PyMuPDF page indices are usually 0-based, so adjust if needed after checking

import fitz  # PyMuPDF
import re

pdf_path = "ed3book_jan26.pdf"

doc = fitz.open(pdf_path)

# Try extracting the printed Chapter 5 page range: 96–119
# If the extracted text looks shifted, we will adjust by checking the first page output
start_page = 104   # 0-based index for printed page 96
end_page = 127    # exclusive upper bound if using range()

chapter_pages = []
for page_num in range(start_page, end_page):
    text = doc[page_num].get_text("text")
    chapter_pages.append(text)

raw_chapter_text = "\n\n".join(chapter_pages)

print("Extracted characters:", len(raw_chapter_text))
print(raw_chapter_text[:2000])

Extracted characters: 68704
5.1
•
LEXICAL SEMANTICS
97
5.1
Lexical Semantics
Let’s begin by introducing some basic principles of word meaning. How should
we represent the meaning of a word? In the n-gram models of Chapter 3, and in
classical NLP applications, our only representation of a word is as a string of letters,
or an index in a vocabulary list. This representation is not that different from a
tradition in philosophy, perhaps you’ve seen it in introductory logic classes, in which
the meaning of words is represented by just spelling the word with small capital
letters; representing the meaning of “dog” as DOG, and “cat” as CAT, or by using an
apostrophe (DOG’).
Representing the meaning of a word by capitalizing it is a pretty unsatisfactory
model. You might have seen a version of a joke due originally to semanticist Barbara
Partee (Carlson, 1977):
Q: What’s the meaning of life?
A: LIFE’
Surely we can do better than this! After all, we’ll want a model of word meaning
to do all sor

In [2]:
import re

def clean_text(text: str) -> str:
    # Remove isolated bullets
    text = re.sub(r'\n\s*[•●]\s*\n', '\n', text)

    # Remove standalone page numbers
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)

    # Remove repeated section markers like "5.1" on their own line
    text = re.sub(r'\n\s*\d+\.\d+\s*\n', '\n', text)

    # Remove repeated all-caps headings if followed again by title case heading
    # Example: LEXICAL SEMANTICS followed by Lexical Semantics
    text = re.sub(r'\n([A-Z][A-Z\s\-]{3,})\n(?=[A-Z][a-z])', '\n', text)

    # Normalize multiple newlines
    text = re.sub(r'\n{2,}', '\n\n', text)

    # Normalize spaces inside lines
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

clean_chapter_text = clean_text(raw_chapter_text)

print("Cleaned characters:", len(clean_chapter_text))
print(clean_chapter_text[:2000])

Cleaned characters: 67690
5.1
Lexical Semantics
Let’s begin by introducing some basic principles of word meaning. How should
we represent the meaning of a word? In the n-gram models of Chapter 3, and in
classical NLP applications, our only representation of a word is as a string of letters,
or an index in a vocabulary list. This representation is not that different from a
tradition in philosophy, perhaps you’ve seen it in introductory logic classes, in which
the meaning of words is represented by just spelling the word with small capital
letters; representing the meaning of “dog” as DOG, and “cat” as CAT, or by using an
apostrophe (DOG’).
Representing the meaning of a word by capitalizing it is a pretty unsatisfactory
model. You might have seen a version of a joke due originally to semanticist Barbara
Partee (Carlson, 1977):
Q: What’s the meaning of life?
A: LIFE’
Surely we can do better than this! After all, we’ll want a model of word meaning
to do all sorts of things for us. It shoul

In [3]:
import re

def clean_text_v2(text: str) -> str:
    # remove isolated bullets
    text = re.sub(r'\n\s*[•●]\s*\n', '\n', text)

    # remove standalone page numbers
    text = re.sub(r'\n\s*\d{1,3}\s*\n', '\n', text)

    # remove standalone section markers like 5.1 / 5.2 / 5.10
    text = re.sub(r'(?m)^\s*\d+\.\d+\s*$', '', text)

    # fix hyphenated line breaks: mean-\nings -> meanings
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)

    # replace remaining newlines inside paragraphs with spaces
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # remove repeated empty lines
    text = re.sub(r'\n{2,}', '\n\n', text)

    # remove very short noisy lines such as "...", "1.", etc.
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        s = line.strip()
        if re.fullmatch(r'\.{2,}', s):
            continue
        if re.fullmatch(r'\d+\.', s):
            continue
        cleaned_lines.append(line)

    text = '\n'.join(cleaned_lines)

    # normalize spaces
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' +\n', '\n', text)

    return text.strip()

clean_chapter_text = clean_text_v2(raw_chapter_text)

print("Cleaned characters:", len(clean_chapter_text))
print(clean_chapter_text[:1500])

Cleaned characters: 68017
LEXICAL SEMANTICS

Lexical Semantics Let’s begin by introducing some basic principles of word meaning. How should we represent the meaning of a word? In the n-gram models of Chapter 3, and in classical NLP applications, our only representation of a word is as a string of letters, or an index in a vocabulary list. This representation is not that different from a tradition in philosophy, perhaps you’ve seen it in introductory logic classes, in which the meaning of words is represented by just spelling the word with small capital letters; representing the meaning of “dog” as DOG, and “cat” as CAT, or by using an apostrophe (DOG’). Representing the meaning of a word by capitalizing it is a pretty unsatisfactory model. You might have seen a version of a joke due originally to semanticist Barbara Partee (Carlson, 1977): Q: What’s the meaning of life? A: LIFE’ Surely we can do better than this! After all, we’ll want a model of word meaning to do all sorts of things f

In [4]:
with open("chapter5_embeddings_clean.txt", "w", encoding="utf-8") as f:
    f.write(clean_chapter_text)

print("Saved cleaned chapter text.")

Saved cleaned chapter text.


In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

chapter_doc = Document(
    page_content=clean_chapter_text,
    metadata={"source": "ed3book_jan26.pdf", "chapter": 5, "title": "Embeddings"}
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

naive_chunks = text_splitter.split_documents([chapter_doc])

print("Number of chunks:", len(naive_chunks))
print("\n--- First chunk preview ---\n")
print(naive_chunks[0].page_content[:1000])

Number of chunks: 117

--- First chunk preview ---

LEXICAL SEMANTICS


That means the splitter is treating the heading as its own tiny chunk.
We should fix that before building FAISS.

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import re

chapter_doc = Document(
    page_content=clean_chapter_text,
    metadata={"source": "ed3book_jan26.pdf", "chapter": 5, "title": "Embeddings"}
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

raw_chunks = text_splitter.split_documents([chapter_doc])

# remove very short / noisy chunks
naive_chunks = []
for chunk in raw_chunks:
    text = chunk.page_content.strip()
    
    # skip tiny chunks
    if len(text) < 120:
        continue
    
    # skip chunks that are basically just headings
    if re.fullmatch(r"[A-Z][A-Z\s\-]+", text):
        continue

    naive_chunks.append(chunk)

print("Raw chunks:", len(raw_chunks))
print("Filtered chunks:", len(naive_chunks))
print("\n--- First chunk preview ---\n")
print(naive_chunks[0].page_content[:1200])

Raw chunks: 94
Filtered chunks: 91

--- First chunk preview ---

Lexical Semantics Let’s begin by introducing some basic principles of word meaning. How should we represent the meaning of a word? In the n-gram models of Chapter 3, and in classical NLP applications, our only representation of a word is as a string of letters, or an index in a vocabulary list. This representation is not that different from a tradition in philosophy, perhaps you’ve seen it in introductory logic classes, in which the meaning of words is represented by just spelling the word with small capital letters; representing the meaning of “dog” as DOG, and “cat” as CAT, or by using an apostrophe (DOG’). Representing the meaning of a word by capitalizing it is a pretty unsatisfactory model. You might have seen a version of a joke due originally to semanticist Barbara Partee (Carlson, 1977): Q: What’s the meaning of life? A: LIFE’ Surely we can do better than this! After all, we’ll want a model of word meaning to do a

In [7]:
# Step 4.1: Build embeddings + FAISS vector store for Naive RAG

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},   
    encode_kwargs={"normalize_embeddings": True}
)

naive_vectorstore = FAISS.from_documents(
    documents=naive_chunks,
    embedding=embedding_model
)

naive_retriever = naive_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Naive FAISS vector store created successfully.")
print("Total indexed chunks:", len(naive_chunks))

/home/noname/Documents/NLU-Assignment/NLP-Assignment-2026/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8134.16it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Naive FAISS vector store created successfully.
Total indexed chunks: 91


In [8]:
query = "What is lexical semantics?"

retrieved_docs = naive_retriever.get_relevant_documents(query)

print(f"Retrieved {len(retrieved_docs)} documents\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"--- Retrieved Doc {i} ---")
    print(doc.page_content[:800])
    print()

Retrieved 3 documents

--- Retrieved Doc 1 ---
. The cosine of two vectors—a normalized dot product—is the most popular such metric. Historical Notes The idea of vector semantics arose out of research in the 1950s in three distinct ﬁelds: linguistics, psychology, and computer science, each of which contributed a fundamental aspect of the model. The idea that meaning is related to the distribution of words in context was widespread in linguistic theory of the 1950s, among distributionalists like Zellig Harris, Martin Joos, and J. R. Firth, and semioticians like Thomas Sebeok. As Joos (1950) put it, the linguist’s “meaning” of a morpheme. . . is by deﬁnition the set of conditional probabilities of its occurrence in context with all other morphemes. The idea that the meaning of a word might be modeled as a point in a multidimensional sema

--- Retrieved Doc 2 ---
Vector Semantics: The Intuition Vector semantics is the standard way to represent word meaning in NLP, helping vector semantics

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"   # use flan-t5-small if base is too heavy

tokenizer = AutoTokenizer.from_pretrained(model_name)
qa_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
qa_model = qa_model.to(device)

print("Model loaded on:", device)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 9806.19it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on: cuda


In [10]:
def generate_answer(prompt, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = qa_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [11]:
def naive_rag_answer(question, k=5):
    docs = naive_retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are answering questions from Chapter 5: Embeddings.

Use only the context below to answer.
Write 2-4 complete sentences.
If the answer is not in the context, say: I cannot find the answer in the chapter.

Context:
{context}

Question: {question}

Answer:"""

    answer = generate_answer(prompt, max_new_tokens=160)

    return {
        "question": question,
        "answer": answer,
        "retrieved_context": context
    }

In [12]:
def naive_rag_answer(question, k=5):
    docs = naive_retriever.get_relevant_documents(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""You are answering questions from Chapter 5: Embeddings.

Use only the context below to answer.
Write 2-4 complete sentences.
If the answer is not in the context, say: I cannot find the answer in the chapter.

Context:
{context}

Question: {question}

Answer:"""

    answer = generate_answer(prompt, max_new_tokens=160)

    return {
        "question": question,
        "answer": answer,
        "retrieved_context": context
    }

In [13]:
result = naive_rag_answer("What is lexical semantics?")

print("Question:", result["question"])
print("\nAnswer:\n", result["answer"])

Question: What is lexical semantics?

Answer:
 standard way to represent word meaning in NLP


In [14]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

local_model_name = "google/flan-t5-small"

local_tokenizer = AutoTokenizer.from_pretrained(local_model_name)
local_model = AutoModelForSeq2SeqLM.from_pretrained(local_model_name)

local_device = "cuda" if torch.cuda.is_available() else "cpu"
local_model = local_model.to(local_device)

print("Local model loaded on:", local_device)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 10873.93it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Local model loaded on: cuda


In [15]:
def enrich_chunk(chunk: str, document: str, title: str) -> str:
    doc_excerpt = document[:2500]

    prompt = f"""
You are writing a short contextual note for a chunk from a textbook chapter.

Chapter title: {title}

Document excerpt:
{doc_excerpt}

Chunk:
{chunk}

Write exactly one sentence that explains what this chunk is mainly about in relation to the chapter.
Start your sentence with:
This chunk from {title} discusses
""".strip()

    inputs = local_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(local_device) for k, v in inputs.items()}

    outputs = local_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    context = local_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if not context.startswith(f"This chunk from {title} discusses"):
        context = f"This chunk from {title} discusses concepts related to word meaning and embeddings."

    if len(context) < 30:
        context = f"This chunk from {title} discusses concepts related to word meaning and embeddings."

    return f"{context}\n\n{chunk}"

In [16]:
def enrich_chunk(chunk: str, document: str, title: str) -> str:
    doc_excerpt = document[:2500]

    prompt = f"""
You are writing a short contextual note for a chunk from a textbook chapter.

Chapter title: {title}

Document excerpt:
{doc_excerpt}

Chunk:
{chunk}

Write exactly one sentence that explains what this chunk is mainly about in relation to the chapter.
Start your sentence with:
This chunk from {title} discusses
""".strip()

    inputs = local_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(local_device) for k, v in inputs.items()}

    outputs = local_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    context = local_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if not context.startswith(f"This chunk from {title} discusses"):
        context = f"This chunk from {title} discusses concepts related to word meaning and embeddings."

    if len(context) < 30:
        context = f"This chunk from {title} discusses concepts related to word meaning and embeddings."

    return f"{context}\n\n{chunk}"

In [17]:
test = enrich_chunk(
    chunk=naive_chunks[0].page_content,
    document=clean_chapter_text,
    title="Embeddings"
)

print(test[:800])

This chunk from Embeddings discusses concepts related to word meaning and embeddings.

Lexical Semantics Let’s begin by introducing some basic principles of word meaning. How should we represent the meaning of a word? In the n-gram models of Chapter 3, and in classical NLP applications, our only representation of a word is as a string of letters, or an index in a vocabulary list. This representation is not that different from a tradition in philosophy, perhaps you’ve seen it in introductory logic classes, in which the meaning of words is represented by just spelling the word with small capital letters; representing the meaning of “dog” as DOG, and “cat” as CAT, or by using an apostrophe (DOG’). Representing the meaning of a word by capitalizing it is a pretty unsatisfactory model. You migh


In [18]:
contextual_chunks = []

for i, doc in enumerate(naive_chunks):
    enriched_text = enrich_chunk(
        chunk=doc.page_content,
        document=clean_chapter_text,
        title="Embeddings"
    )

    contextual_chunks.append(
        Document(
            page_content=enriched_text,
            metadata=doc.metadata
        )
    )

    print(f"Enriched chunk {i+1}/{len(naive_chunks)}")

Enriched chunk 1/91
Enriched chunk 2/91
Enriched chunk 3/91
Enriched chunk 4/91
Enriched chunk 5/91
Enriched chunk 6/91
Enriched chunk 7/91
Enriched chunk 8/91
Enriched chunk 9/91
Enriched chunk 10/91
Enriched chunk 11/91
Enriched chunk 12/91
Enriched chunk 13/91
Enriched chunk 14/91
Enriched chunk 15/91
Enriched chunk 16/91
Enriched chunk 17/91
Enriched chunk 18/91
Enriched chunk 19/91
Enriched chunk 20/91
Enriched chunk 21/91
Enriched chunk 22/91
Enriched chunk 23/91
Enriched chunk 24/91
Enriched chunk 25/91
Enriched chunk 26/91
Enriched chunk 27/91
Enriched chunk 28/91
Enriched chunk 29/91
Enriched chunk 30/91
Enriched chunk 31/91
Enriched chunk 32/91
Enriched chunk 33/91
Enriched chunk 34/91
Enriched chunk 35/91
Enriched chunk 36/91
Enriched chunk 37/91
Enriched chunk 38/91
Enriched chunk 39/91
Enriched chunk 40/91
Enriched chunk 41/91
Enriched chunk 42/91
Enriched chunk 43/91
Enriched chunk 44/91
Enriched chunk 45/91
Enriched chunk 46/91
Enriched chunk 47/91
Enriched chunk 48/91
E

In [19]:
contextual_vectorstore = FAISS.from_documents(
    documents=contextual_chunks,
    embedding=embedding_model
)

contextual_retriever = contextual_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Local contextual retriever built successfully.")

Local contextual retriever built successfully.


In [20]:
qa_pairs = [
    {
        "question": "What is lexical semantics?",
        "reference_answer": "Lexical semantics is the linguistic study of word meaning. It focuses on the basic principles of word meaning and how words represent meaning in language."
    },
    {
        "question": "Why is representing word meaning as just a word label unsatisfactory?",
        "reference_answer": ""
    },
    {
        "question": "What kinds of information should a model of word meaning capture?",
        "reference_answer": ""
    },
    {
        "question": "What is a word sense?",
        "reference_answer": ""
    },
    {
        "question": "What is synonymy?",
        "reference_answer": ""
    },
    {
        "question": "What is antonymy?",
        "reference_answer": ""
    },
    {
        "question": "What is vector semantics?",
        "reference_answer": ""
    },
    {
        "question": "How does distribution help define meaning?",
        "reference_answer": ""
    },
    {
        "question": "What are count-based embeddings?",
        "reference_answer": ""
    },
    {
        "question": "How are word-context matrices used in embeddings?",
        "reference_answer": ""
    },
    {
        "question": "Why are dense embeddings preferred over sparse vectors?",
        "reference_answer": ""
    },
    {
        "question": "What is cosine similarity?",
        "reference_answer": ""
    },
    {
        "question": "Why is cosine similarity useful for embeddings?",
        "reference_answer": ""
    },
    {
        "question": "What is word2vec?",
        "reference_answer": ""
    },
    {
        "question": "What is the basic idea behind CBOW?",
        "reference_answer": ""
    },
    {
        "question": "What is the basic idea behind Skip-gram?",
        "reference_answer": ""
    },
    {
        "question": "How can embeddings be visualized?",
        "reference_answer": ""
    },
    {
        "question": "What semantic properties can embeddings capture?",
        "reference_answer": ""
    },
    {
        "question": "What is bias in embeddings?",
        "reference_answer": ""
    },
    {
        "question": "How can vector models be evaluated?",
        "reference_answer": ""
    }
]

print("Total QA pairs:", len(qa_pairs))

Total QA pairs: 20


In [21]:
qa_pairs[1]["reference_answer"] = "Representing meaning as just a word label is unsatisfactory because it does not capture relationships between words, differences in connotation, or the multiple senses that a word can have."

qa_pairs[2]["reference_answer"] = "A model of word meaning should capture similarity, opposition, connotation, and different perspectives on related events. It should also support inference about how words are related in meaning."

qa_pairs[3]["reference_answer"] = "A word sense is one particular meaning of a word. A single word can have multiple senses, which is why polysemy can make interpretation difficult."

qa_pairs[4]["reference_answer"] = "Synonymy is the relationship between words or senses that have the same or very similar meanings."

qa_pairs[5]["reference_answer"] = "Antonymy is the relationship between words with opposite meanings, such as hot and cold."

In [22]:
qa_pairs[6]["reference_answer"] = "Vector semantics is a way of representing word meaning as vectors in a multidimensional space. Words with similar meanings are placed near each other because they occur in similar contexts."

qa_pairs[7]["reference_answer"] = "Distribution helps define meaning by assuming that words occurring in similar contexts tend to have similar meanings. This idea is often summarized as meaning being derived from patterns of use."

qa_pairs[8]["reference_answer"] = "Count-based embeddings are vector representations built from counting how often words co-occur with other words or contexts in a corpus. These counts are then used to represent words numerically."

qa_pairs[9]["reference_answer"] = "Word-context matrices represent words as rows and contexts as columns, with values based on co-occurrence information. These matrices are used to build vector representations of meaning from corpus statistics."

qa_pairs[10]["reference_answer"] = "Dense embeddings are preferred over sparse vectors because they are more compact and can better capture semantic similarities between words. They also generalize better and are more useful in downstream NLP tasks."

In [23]:
for i in range(6, 11):
    print(i, qa_pairs[i]["question"])
    print(qa_pairs[i]["reference_answer"])
    print()

6 What is vector semantics?
Vector semantics is a way of representing word meaning as vectors in a multidimensional space. Words with similar meanings are placed near each other because they occur in similar contexts.

7 How does distribution help define meaning?
Distribution helps define meaning by assuming that words occurring in similar contexts tend to have similar meanings. This idea is often summarized as meaning being derived from patterns of use.

8 What are count-based embeddings?
Count-based embeddings are vector representations built from counting how often words co-occur with other words or contexts in a corpus. These counts are then used to represent words numerically.

9 How are word-context matrices used in embeddings?
Word-context matrices represent words as rows and contexts as columns, with values based on co-occurrence information. These matrices are used to build vector representations of meaning from corpus statistics.

10 Why are dense embeddings preferred over sp

In [24]:
qa_pairs[11]["reference_answer"] = "Cosine similarity is a measure of how similar two vectors are based on the angle between them. It is computed from the normalized dot product of the vectors."

qa_pairs[12]["reference_answer"] = "Cosine similarity is useful for embeddings because it measures how close two word vectors are in direction, which helps identify semantic similarity between words."

qa_pairs[13]["reference_answer"] = "Word2vec is a method for learning dense word embeddings from large corpora. It learns word representations by predicting words from their contexts or contexts from a target word."

qa_pairs[14]["reference_answer"] = "CBOW, or Continuous Bag of Words, predicts a target word from the surrounding context words. It learns embeddings by using nearby words to infer the missing center word."

qa_pairs[15]["reference_answer"] = "Skip-gram predicts surrounding context words from a target word. It learns embeddings by using one word to predict the words that are likely to appear near it."

In [25]:
for i in range(11, 16):
    print(i, qa_pairs[i]["question"])
    print(qa_pairs[i]["reference_answer"])
    print()

11 What is cosine similarity?
Cosine similarity is a measure of how similar two vectors are based on the angle between them. It is computed from the normalized dot product of the vectors.

12 Why is cosine similarity useful for embeddings?
Cosine similarity is useful for embeddings because it measures how close two word vectors are in direction, which helps identify semantic similarity between words.

13 What is word2vec?
Word2vec is a method for learning dense word embeddings from large corpora. It learns word representations by predicting words from their contexts or contexts from a target word.

14 What is the basic idea behind CBOW?
CBOW, or Continuous Bag of Words, predicts a target word from the surrounding context words. It learns embeddings by using nearby words to infer the missing center word.

15 What is the basic idea behind Skip-gram?
Skip-gram predicts surrounding context words from a target word. It learns embeddings by using one word to predict the words that are likely

In [26]:
qa_pairs[16]["reference_answer"] = "Embeddings can be visualized by reducing their dimensions to two or three dimensions using visualization techniques. This helps reveal clusters, relationships, and patterns among words."

qa_pairs[17]["reference_answer"] = "Embeddings can capture semantic properties such as similarity, relatedness, analogical relationships, and groupings of words with shared meanings or functions."

qa_pairs[18]["reference_answer"] = "Bias in embeddings refers to the fact that word vectors can reflect social and cultural biases present in the training data. As a result, embeddings may encode stereotypes or unfair associations."

qa_pairs[19]["reference_answer"] = "Vector models can be evaluated by testing how well they capture semantic similarity, analogical relationships, or performance on downstream NLP tasks. Evaluation can be intrinsic or extrinsic depending on the goal."

In [27]:
for i, qa in enumerate(qa_pairs):
    print(i, qa["question"])
    print(qa["reference_answer"])
    print()

0 What is lexical semantics?
Lexical semantics is the linguistic study of word meaning. It focuses on the basic principles of word meaning and how words represent meaning in language.

1 Why is representing word meaning as just a word label unsatisfactory?
Representing meaning as just a word label is unsatisfactory because it does not capture relationships between words, differences in connotation, or the multiple senses that a word can have.

2 What kinds of information should a model of word meaning capture?
A model of word meaning should capture similarity, opposition, connotation, and different perspectives on related events. It should also support inference about how words are related in meaning.

3 What is a word sense?
A word sense is one particular meaning of a word. A single word can have multiple senses, which is why polysemy can make interpretation difficult.

4 What is synonymy?
Synonymy is the relationship between words or senses that have the same or very similar meanings

In [28]:
import re

def extractive_fallback_answer(question, retriever, k=5):
    docs = retriever.get_relevant_documents(question)

    cleaned_docs = []
    for doc in docs:
        text = doc.page_content

        # remove contextual prefix if present
        text = re.sub(
            r"^This chunk from Embeddings discusses.*?\.\s*",
            "",
            text
        )

        # remove citation-like clutter
        text = re.sub(r'\([^)]*\d{4}[^)]*\)', ' ', text)

        # normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        cleaned_docs.append(text)

    context = " ".join(cleaned_docs)
    sentences = re.split(r'(?<=[.!?])\s+', context)
    query_words = [w.lower() for w in re.findall(r'\w+', question) if len(w) > 2]

    scored = []
    for i, sent in enumerate(sentences):
        s = sent.strip()
        if not s:
            continue

        s_lower = s.lower()
        score = 0

        for w in query_words:
            if w in s_lower:
                score += 1

        if question.lower() in s_lower:
            score += 10

        if "is called" in s_lower or "is the study of" in s_lower:
            score += 3

        score += max(0, 3 - i * 0.1)
        scored.append((score, s))

    scored.sort(key=lambda x: x[0], reverse=True)

    best_sentences = []
    seen = set()
    for score, sent in scored:
        if score <= 0:
            continue
        if sent not in seen:
            best_sentences.append(sent)
            seen.add(sent)
        if len(best_sentences) == 2:
            break

    if not best_sentences:
        return "I cannot find the answer in the chapter."

    return " ".join(best_sentences)

In [ ]:
naive_results = []
contextual_results = []

for qa in qa_pairs:
    question = qa["question"]
    ref = qa["reference_answer"]

    naive_ans = extractive_fallback_answer(question, naive_retriever, k=5)
    contextual_ans = extractive_fallback_answer(question, contextual_retriever, k=5)

    naive_results.append({
        "question": question,
        "reference_answer": ref,
        "generated_answer": naive_ans
    })

    contextual_results.append({
        "question": question,
        "reference_answer": ref,
        "generated_answer": contextual_ans
    })

print("Naive results:", len(naive_results))
print("Contextual results:", len(contextual_results))
print("\nSample naive answer:\n", naive_results[0])
print("\nSample contextual answer:\n", contextual_results[0])

Naive results: 20
Contextual results: 20

Sample naive answer:
 {'question': 'What is lexical semantics?', 'reference_answer': 'Lexical semantics is the linguistic study of word meaning. It focuses on the basic principles of word meaning and how words represent meaning in language.', 'generated_answer': 'Historical Notes The idea of vector semantics arose out of research in the 1950s in three distinct ﬁelds: linguistics, psychology, and computer science, each of which contributed a fundamental aspect of the model. .'}

Sample contextual answer:
 {'question': 'What is lexical semantics?', 'reference_answer': 'Lexical semantics is the linguistic study of word meaning. It focuses on the basic principles of word meaning and how words represent meaning in language.', 'generated_answer': 'Historical Notes The idea of vector semantics arose out of research in the 1950s in three distinct ﬁelds: linguistics, psychology, and computer science, each of which contributed a fundamental aspect of the

In [30]:
from rouge_score import rouge_scorer
import pandas as pd

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def evaluate_rouge(results):
    scores = []

    for item in results:
        reference = item["reference_answer"]
        generated = item["generated_answer"]

        score = scorer.score(reference, generated)

        scores.append({
            "question": item["question"],
            "rouge1_f": score["rouge1"].fmeasure,
            "rouge2_f": score["rouge2"].fmeasure,
            "rougeL_f": score["rougeL"].fmeasure
        })

    df = pd.DataFrame(scores)

    avg_scores = {
        "rouge1_f": df["rouge1_f"].mean(),
        "rouge2_f": df["rouge2_f"].mean(),
        "rougeL_f": df["rougeL_f"].mean()
    }

    return df, avg_scores

In [31]:
naive_df, naive_avg = evaluate_rouge(naive_results)
contextual_df, contextual_avg = evaluate_rouge(contextual_results)

print("Naive RAG Average ROUGE Scores:")
print(naive_avg)

print("\nContextual Retrieval Average ROUGE Scores:")
print(contextual_avg)

Naive RAG Average ROUGE Scores:
{'rouge1_f': 0.24996737910008707, 'rouge2_f': 0.04366050661552578, 'rougeL_f': 0.16045067986987155}

Contextual Retrieval Average ROUGE Scores:
{'rouge1_f': 0.23156311038539937, 'rouge2_f': 0.037733429641898866, 'rougeL_f': 0.14716023654801513}


In [32]:
comparison_df = pd.DataFrame([
    {
        "method": "Naive RAG",
        "ROUGE-1": naive_avg["rouge1_f"],
        "ROUGE-2": naive_avg["rouge2_f"],
        "ROUGE-L": naive_avg["rougeL_f"]
    },
    {
        "method": "Contextual Retrieval",
        "ROUGE-1": contextual_avg["rouge1_f"],
        "ROUGE-2": contextual_avg["rouge2_f"],
        "ROUGE-L": contextual_avg["rougeL_f"]
    }
])

comparison_df

,method,ROUGE-1,ROUGE-2,ROUGE-L
0,Naive RAG,0.249967,0.043661,0.160451
1,Contextual Retrieval,0.231563,0.037733,0.147160


In [33]:
import json

final_output = {
    "student_id": "126425",
    "chapter": {
        "chapter_number": 5,
        "chapter_title": "Embeddings"
    },
    "evaluation_summary": {
        "naive_rag": {
            "rouge1": float(naive_avg["rouge1_f"]),
            "rouge2": float(naive_avg["rouge2_f"]),
            "rougeL": float(naive_avg["rougeL_f"])
        },
        "contextual_retrieval": {
            "rouge1": float(contextual_avg["rouge1_f"]),
            "rouge2": float(contextual_avg["rouge2_f"]),
            "rougeL": float(contextual_avg["rougeL_f"])
        }
    },
    "qa_results": []
}

for i in range(len(qa_pairs)):
    final_output["qa_results"].append({
        "question": qa_pairs[i]["question"],
        "reference_answer": qa_pairs[i]["reference_answer"],
        "naive_rag_answer": naive_results[i]["generated_answer"],
        "contextual_retrieval_answer": contextual_results[i]["generated_answer"]
    })

with open("assignment_output_126425.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print("Saved assignment_output_126425.json")

Saved assignment_output_126425.json


In [34]:
print(final_output["qa_results"][0])

{'question': 'What is lexical semantics?', 'reference_answer': 'Lexical semantics is the linguistic study of word meaning. It focuses on the basic principles of word meaning and how words represent meaning in language.', 'naive_rag_answer': 'Historical Notes The idea of vector semantics arose out of research in the 1950s in three distinct ﬁelds: linguistics, psychology, and computer science, each of which contributed a fundamental aspect of the model. .', 'contextual_retrieval_answer': 'Historical Notes The idea of vector semantics arose out of research in the 1950s in three distinct ﬁelds: linguistics, psychology, and computer science, each of which contributed a fundamental aspect of the model. .'}


In [35]:
contextual_vectorstore.save_local("contextual_faiss_index")
print("Saved contextual FAISS index.")

Saved contextual FAISS index.
